# 🚀 Clasificador PAES con Ollama + GPU RTX 2050

**Objetivos:**
1. Leer JSONs intermedios (desafios, materia, mixto)
2. Clasificar contenido mixto con Ollama/Qwen
3. Poblar PostgreSQL con preguntas + materia

**Requiere:**
- Ollama corriendo localmente (`ollama serve`)
- Modelo Qwen descargado (`ollama pull qwen:7b`)
- PostgreSQL accesible

## 1️⃣ Configuración Inicial

In [ ]:
import json
import requests
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any
from tqdm import tqdm
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Rutas
PROCESSED_DATA_DIR = Path('/home/gabriel/procesamiento_paes/processed_data')
OLLAMA_URL = 'http://localhost:11434'  # URL local de Ollama
MODEL = 'qwen:7b'  # Puedes cambiar a qwen:14b si tienes VRAM

print(f"✓ Rutas configuradas")
print(f"  Datos en: {PROCESSED_DATA_DIR}")
print(f"  Ollama: {OLLAMA_URL}")
print(f"  Modelo: {MODEL}")

## 2️⃣ Cargar Datos Intermedios

In [ ]:
def load_jsonl(filepath: str) -> List[Dict[str, Any]]:
    """Lee archivo JSONL."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Cargar datos generados ayer
desafios = load_jsonl(PROCESSED_DATA_DIR / 'desafios_preguntas.jsonl')
materia = load_jsonl(PROCESSED_DATA_DIR / 'materia_estudio.jsonl')
mixto = load_jsonl(PROCESSED_DATA_DIR / 'contenido_mixto.jsonl')

print(f"✅ Datos cargados:")
print(f"   - {len(desafios)} desafíos/preguntas limpias")
print(f"   - {len(materia)} archivos de materia de estudio")
print(f"   - {len(mixto)} archivos con contenido mixto (para clasificar)")

## 3️⃣ Ejemplo: Una Pregunta Limpia

In [ ]:
if desafios:
    ejemplo = desafios[0]
    print(f"Pregunta #{ejemplo['numero']}:")
    print(f"{ejemplo['enunciado'][:150]}...")
    print(f"\nOpciones:")
    for opt in ejemplo['opciones']:
        marca = '✓' if opt['es_correcta'] else ' '
        print(f"  [{marca}] {opt['label']}) {opt['texto'][:70]}")

## 4️⃣ Verificar Ollama

In [ ]:
# Verifica que Ollama está corriendo
try:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    if response.status_code == 200:
        models = response.json().get('models', [])
        print(f"✅ Ollama está corriendo")
        print(f"   Modelos disponibles: {[m['name'] for m in models]}")
    else:
        print(f"❌ Error: Ollama devolvió {response.status_code}")
except Exception as e:
    print(f"❌ Ollama NO está corriendo. Inicia con: ollama serve")
    print(f"   Error: {e}")

## 5️⃣ Clasificar Contenido Mixto con Qwen

In [ ]:
def classify_content_with_ollama(texto: str, modelo: str = 'qwen:7b') -> Dict[str, Any]:
    """
    Usa Ollama para clasificar si el texto es:
    - PREGUNTA: ejercicio/desafío con opciones
    - MATERIA: explicación/teoría/fórmula
    - MIXTO: contiene ambos
    """
    prompt = f"""Clasifica si este texto es principalmente PREGUNTA, MATERIA o MIXTO.
PREGUNTA: contiene numeración, opciones A/B/C/D/E
MATERIA: contiene explicaciones, teoría, definiciones
MIXTO: contiene ambas

Responde SOLO UNA PALABRA: PREGUNTA, MATERIA o MIXTO

Texto:
{texto[:500]}

Clasificación:"""
    
    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": modelo,
                "prompt": prompt,
                "stream": False,
                "temperature": 0.1  # Bajo para respuestas determinísticas
            },
            timeout=30
        )
        resultado = response.json().get('response', '').strip()
        
        # Normaliza respuesta
        if 'PREGUNTA' in resultado.upper():
            return {'tipo': 'PREGUNTA', 'confidence': 0.9}
        elif 'MATERIA' in resultado.upper():
            return {'tipo': 'MATERIA', 'confidence': 0.9}
        elif 'MIXTO' in resultado.upper():
            return {'tipo': 'MIXTO', 'confidence': 0.8}
        else:
            return {'tipo': 'DESCONOCIDO', 'confidence': 0.3, 'raw': resultado}
    except Exception as e:
        return {'tipo': 'ERROR', 'error': str(e)}

# Prueba con el primer documento mixto
if mixto:
    test_texto = mixto[0]['paginas'][0]['texto'] if mixto[0].get('paginas') else ""
    print(f"Clasificando: {mixto[0].get('archivo', 'unknown')[:50]}...")
    resultado = classify_content_with_ollama(test_texto)
    print(f"Resultado: {resultado}")

## 6️⃣ Procesar TODO el Contenido Mixto (⚠️ LENTO - ~15-30 min)

In [ ]:
# SKIP si solo quieres probar. Descomenta para ejecutar full:
"""
clasificaciones = []
for doc in tqdm(mixto, desc="Clasificando contenido mixto"):
    for pagina in doc.get('paginas', []):
        texto = pagina.get('texto', '')[:1000]
        if texto.strip():
            clasificacion = classify_content_with_ollama(texto)
            clasificaciones.append({
                'archivo': doc['archivo'],
                'pagina': pagina.get('numero'),
                'tipo': clasificacion['tipo'],
                'confidence': clasificacion.get('confidence', 0)
            })

# Guarda resultados
with open(PROCESSED_DATA_DIR / 'clasificaciones_ollama.jsonl', 'w') as f:
    for c in clasificaciones:
        f.write(json.dumps(c) + '\\n')

print(f"✅ Clasificadas {len(clasificaciones)} páginas")
"""

print("️⏭️  Salta este paso por ahora (muy lento)")

## 7️⃣ Cargar en PostgreSQL

In [ ]:
from sqlalchemy import create_engine, select, text
from sqlalchemy.orm import sessionmaker
import sys

# Configura ruta del backend
sys.path.insert(0, '/home/gabriel/proyectos/Mvp-paes2/backend')

try:
    from app.core.config import settings
    from app.db.models import Exam, Subject, Topic, Question, QuestionChoice
    from app.db.session import SessionLocal
    
    # Conecta a BD
    db = SessionLocal()
    
    # Verifica que PAES exam existe
    exam = db.scalar(select(Exam).where(Exam.code == settings.PAES_CODE))
    print(f"✅ Conectado a PostgreSQL")
    print(f"   PAES exam ID: {exam.id if exam else 'NO EXISTE'}")
    
    # Lista asignaturas
    subjects = db.scalar(select(Subject))
    print(f"   Asignaturas: {subjects}")
    
except Exception as e:
    print(f"❌ Error de BD: {e}")

## 8️⃣ Insertar Desafíos en BD

In [ ]:
def load_desafios_to_db(preguntas: List[Dict], db_session) -> int:
    """
    Carga preguntas de desafíos en PostgreSQL.
    """
    from app.db.models import Exam, Subject, Topic, Question, QuestionChoice
    
    exam = db_session.scalar(select(Exam).where(Exam.code='PAES'))
    if not exam:
        print("❌ PAES exam no existe")
        return 0
    
    # Obtiene o crea subject/topic para DESAFIOS
    subject = db_session.scalar(
        select(Subject).where(
            Subject.exam_id == exam.id,
            Subject.code == 'DESAFIOS'
        )
    )
    if not subject:
        subject = Subject(exam_id=exam.id, code='DESAFIOS', name='Desafíos PAES')
        db_session.add(subject)
        db_session.flush()
    
    topic = db_session.scalar(
        select(Topic).where(
            Topic.subject_id == subject.id,
            Topic.code == 'DESAFIOS'
        )
    )
    if not topic:
        topic = Topic(subject_id=subject.id, code='DESAFIOS', name='Desafíos Variados')
        db_session.add(topic)
        db_session.flush()
    
    # Inserta preguntas
    loaded = 0
    for p in tqdm(preguntas, desc="Cargando desafíos"):
        # Evita duplicados
        existing = db_session.scalar(
            select(Question).where(
                Question.topic_id == topic.id,
                Question.prompt == p['enunciado'][:200]
            )
        )
        if existing:
            continue
        
        q = Question(
            topic_id=topic.id,
            prompt=p['enunciado'][:1000],
            question_type='mcq',
            difficulty=2,
            is_active=True
        )
        db_session.add(q)
        db_session.flush()
        
        # Opciones
        for opt in p['opciones']:
            qc = QuestionChoice(
                question_id=q.id,
                label=opt['label'],
                text=opt['texto'][:500],
                is_correct=opt['es_correcta']
            )
            db_session.add(qc)
        
        loaded += 1
    
    db_session.commit()
    return loaded

# Ejecuta carga (comentado para no ejecutar sin datos preparados)
# loaded = load_desafios_to_db(desafios, db)
# print(f"✅ {loaded} preguntas cargadas en PostgreSQL")

print("⏭️  Carga lista (descomenta para ejecutar)")

## 9️⃣ Estadísticas Finales

In [ ]:
import pandas as pd

stats = pd.DataFrame({
    'Categoría': ['Desafíos (limpios)', 'Materia de estudio', 'Contenido mixto (para clasificar)'],
    'Cantidad': [len(desafios), len(materia), len(mixto)],
    'Estado': ['✅ Listo insertar', '⏳ Pendiente estructura', '⏳ Pendiente Ollama']
})

print("\n📊 RESUMEN DE PROCESAMIENTO:\n")
print(stats.to_string(index=False))
print("\n✨ Listo para llenar tu BD de preguntas + materia mañana!")

In [ ]:
# PRÓXIMOS PASOS:
next_steps = """
PLAN PARA MAÑANA (con GPU 2050):

1. ✅ Cargar JSONs preparados (ya hecho)

2. 🚀 Ejecutar clasificación Ollama (sección 6)
   - Toma ~15-30 min según GPU
   - Separa preguntas de materia del contenido mixto

3. 💾 Insertar en PostgreSQL (sección 7-8)
   - Desafíos: 79 preguntas ya listas
   - Mixto clasificado: N preguntas + M archivos materia
   - Materia: 7 archivos de estudio

4. 🎓 Resultado final:
   - DB poblada con preguntas variadas
   - Material de estudio por tópico
   - Frontend listo para usar
"""

print(next_steps)